# NDgpu — CPU vs GPU speed-up on reactor benchmarks (Google Colab)

Times the **same** NDgpu eigenvalue solve on CPU (NumPy) and GPU (CuPy) for
four reactor problems spanning the solver's regimes:

| case | mesh | groups | dim |
|---|---|---|---|
| **C5G7** MOX lattice | Cartesian | 7 (+ upscatter) | 2D |
| **IAEA-3D** | Cartesian + active mask | 2 | 3D |
| **VVER-440** | body-fitted triangular | 2 | 2D |
| **HP-MR** microreactor | triangular prisms | 2 | 3D |

Because CuPy mirrors the NumPy API, the physics code is byte-for-byte identical
on both backends — `device='cpu'|'gpu'` is the only change, and both must return
the same `k_eff` (asserted below).

**To run:** Runtime → Change runtime type → **T4 GPU**, then Run all. Upload
`dist/ndgpu-src.zip` when prompted. The CPU legs take ~1–2 min in total.

In [ ]:
from google.colab import files
uploaded = files.upload()          # upload dist/ndgpu-src.zip
zip_name = next(iter(uploaded))
%pip install -q {zip_name}
try:
    import cupy
except ImportError:
    %pip install -q cupy-cuda12x
!nvidia-smi -L

## Method

Each case is solved once as an untimed **warm-up** (this compiles the CUDA
kernels and allocates the device buffers), then solved again for the timed
number. `solve_seconds` is measured across a device synchronize, so it is
honest GPU wall-clock time — not an unsynchronized launch. All four cases use
the same tolerance; the *speed-up ratio* is independent of the tolerance
because CPU and GPU run the identical iteration sequence.

In [ ]:
import numpy as np
from ndgpu import DiffusionEigenSolver
from ndgpu.tri import TriDiffusionEigenSolver
from ndgpu.benchmarks import build_c5g7_2d, build_iaea, build_vver440, build_hpmr3d

TOL = dict(tol_k=1e-6, tol_source=1e-5)

# Each builder returns (solver, n_unknown_cells, n_groups) for a given device.
def make_c5g7(dev):
    p = build_c5g7_2d(cells_per_pin=4)                 # 204x204 x 7 groups
    return (DiffusionEigenSolver(p.grid, p.materials, p.material_map, bc=p.bc,
                                 device=dev), p.grid.n_cells, p.materials[0].n_groups)

def make_iaea(dev):
    p = build_iaea(cells_per_node=3)                   # 3D, active-masked core
    return (DiffusionEigenSolver(p.grid, p.materials, p.material_map, bc=p.bc,
                                 active=p.active, mask_bc=p.mask_bc, device=dev),
            int(p.active.sum()), 2)

def make_vver(dev):
    p = build_vver440(refine=4)                        # body-fitted triangular hex core
    return (TriDiffusionEigenSolver(p.grid, p.materials, p.material_map,
                                    active=p.active, mask_bc=p.mask_bc, device=dev),
            int(p.active.sum()), 2)

def make_hpmr(dev):
    p = build_hpmr3d(refine=4, nz=20)                  # extruded triangular prisms
    return (TriDiffusionEigenSolver(p.grid, p.materials, p.material_map,
                                    active=p.active, mask_bc=p.mask_bc, bc=p.bc,
                                    device=dev), int(p.active.sum()), 2)

CASES = [("C5G7 2D (7g)", make_c5g7), ("IAEA-3D (2g)", make_iaea),
         ("VVER-440 2D (2g)", make_vver), ("HP-MR 3D (2g)", make_hpmr)]

def bench(make):
    out = {}
    for dev in ("cpu", "gpu"):
        solver, ncells, G = make(dev)
        solver.solve(max_outer=3, tol_k=0.0)           # warm-up (not timed)
        out[dev] = solver.solve(**TOL)
        out["ncells"], out["G"] = ncells, G
    return out

results = []
hdr = (f"{'case':18s}{'unknowns':>11}{'k(cpu)':>10}{'k(gpu)':>10}"
       f"{'cpu [s]':>9}{'gpu [s]':>9}{'speed-up':>9}")
print(hdr); print("-" * len(hdr))
for label, make in CASES:
    b = bench(make)
    kc, kg = b["cpu"].k_eff, b["gpu"].k_eff
    tc, tg = b["cpu"].solve_seconds, b["gpu"].solve_seconds
    assert abs(kc - kg) < 1e-5, f"{label}: CPU/GPU k disagree ({kc} vs {kg})"
    unk = b["ncells"] * b["G"]
    results.append((label, unk, tc, tg, tc / tg))
    print(f"{label:18s}{unk:>11,}{kc:>10.5f}{kg:>10.5f}{tc:>9.2f}{tg:>9.2f}{tc/tg:>8.1f}x")

## Speed-up at a glance

Every bar is the **same solver on the same problem**, CPU vs GPU. The dashed
line is parity (1x); anything to its right is the GPU winning.

In [ ]:
import matplotlib.pyplot as plt

labels = [r[0] for r in results]
speed  = [r[4] for r in results]
fig, ax = plt.subplots(figsize=(7, 3.6))
bars = ax.barh(labels, speed, color="#3987e5")
ax.invert_yaxis()
ax.axvline(1, color="0.6", ls="--", lw=1)
for bar, s in zip(bars, speed):
    ax.text(s, bar.get_y() + bar.get_height() / 2, f" {s:.1f}x",
            va="center", fontsize=10)
ax.set_xlabel("GPU speed-up over CPU")
ax.set_title("NDgpu — identical solve, CPU vs GPU (Colab T4)")
fig.tight_layout(); plt.show()

## Speed-up grows with problem size (and with float32)

GPUs win by keeping thousands of cores fed, so the advantage widens as the mesh
grows. Below, the HP-MR core is refined in-plane and axially; `dtype=float32`
roughly doubles GPU throughput again, at ~1e-6 accuracy in `k_eff` (the value is
unchanged to 5 digits). This is the longest cell — a couple of minutes on CPU.

In [ ]:
import numpy as np

hdr = (f"{'HP-MR mesh':>14}{'unknowns':>11}{'cpu [s]':>9}"
       f"{'gpu64 [s]':>10}{'gpu32 [s]':>10}{'speed-up(32)':>13}")
print(hdr); print("-" * len(hdr))
for refine, nz in [(4, 10), (4, 20), (5, 20)]:
    p = build_hpmr3d(refine=refine, nz=nz)
    unk = int(p.active.sum()) * 2
    t = {}
    for dev, dt in [("cpu", np.float64), ("gpu", np.float64), ("gpu", np.float32)]:
        s = TriDiffusionEigenSolver(p.grid, p.materials, p.material_map,
                                    active=p.active, mask_bc=p.mask_bc, bc=p.bc,
                                    device=dev, dtype=dt)
        s.solve(max_outer=3, tol_k=0.0)
        t[dev + ("32" if dt is np.float32 else "")] = s.solve(**TOL).solve_seconds
    print(f"r={refine} nz={nz:<3}{unk:>17,}{t['cpu']:>9.2f}"
          f"{t['gpu']:>10.2f}{t['gpu32']:>10.2f}{t['cpu'] / t['gpu32']:>12.1f}x")

### Notes

- **Same answer, faster.** The `k(cpu) == k(gpu)` assertion in the table is the
  point of the one-code-path design: the CPU test suite exercises byte-for-byte
  the kernels that run on the GPU.
- **Where the GPU wins.** Small/coarse meshes are dominated by Python and kernel
  launch overhead and can be CPU-faster; the crossover is well below the sizes
  here. The stencil apply is memory-bandwidth bound, so the speed-up tracks the
  GPU's bandwidth advantage over the CPU.
- **VVER-440** is outer-iteration-bound (its 2D bare core has a high dominance
  ratio), so its solve is many cheap power-iteration steps — still GPU-friendly,
  but a different regime from the bandwidth-bound 3D cases.
- **Real cross sections.** These runs use the built-in placeholder/benchmark
  constants; swap in the 11-group ENDF/B-8 HP-MR set with
  `build_hpmr3d(..., materials=hpmr_endfb8_materials(xs_path))`.